# DomainNet (Clipart) Sweep

Parametric p_out sweep on DomainNet clipart (345 classes, 64×64 RGB, 4 semantic clusters) to map the I(α, β) interaction surface.

**Requirements**: A100 GPU runtime on Google Colab.

## Persistent dataset preparation
Run all cells in order. The dataset and outputs remain in Google Drive across Colab runtimes.

**Resume behaviour**: `--resume` skips completed runs (those with `summary.json`) and
resumes interrupted runs from their last checkpoint. Checkpoints, results, and the git
commit hash used for the sweep are all persisted on Drive. After a Colab disconnect,
mount the same Drive and click **Run all** to continue.

Preparation checks the official cleaned split lists and the actual image paths on every run.
A complete cache skips downloads and copies; scanning Drive can still take time.
Missing or zero-byte images are restored from extracted files or the cached ZIP. If necessary,
the official archive is downloaded. Unexpected class folders and files are moved to
`superadditivity_data/domainnet/_repair_backups/`, not deleted.

Analysis and visualization skip automatically when their inputs haven't changed.
Pass `--force` to regenerate them.

Sources: [official cleaned DomainNet downloads](https://ai.bu.edu/DomainNet/).

In [ ]:
DATA_READY = False
DATA_VALIDATED = False
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> A100 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
import sys
import json as _json

REPO = '/content/quantify_superadditivity'
DRIVE_OUT = '/content/drive/MyDrive/superadditivity_outputs'
SWEEP_STATE = os.path.join(DRIVE_OUT, 'pout_sweep_domainnet', '.sweep_state.json')

def _read_sweep_state():
    try:
        with open(SWEEP_STATE) as f:
            return _json.load(f)
    except (OSError, ValueError):
        return {}

def _write_sweep_state(state):
    os.makedirs(os.path.dirname(SWEEP_STATE), exist_ok=True)
    tmp = SWEEP_STATE + '.tmp'
    with open(tmp, 'w') as f:
        _json.dump(state, f, indent=2)
    os.replace(tmp, SWEEP_STATE)

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone',
                    'https://github.com/angelafoua/quantify_superadditivity', REPO], check=True)
os.chdir(REPO)

# If a sweep is already in progress, stay on the same commit.
# Otherwise fetch the latest main and record it.
sweep_state = _read_sweep_state()
pinned_commit = sweep_state.get('git_commit')

subprocess.run(['git', 'fetch', 'origin', 'main'], check=True)

if pinned_commit:
    current = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
    if current != pinned_commit:
        print(f'Restoring pinned commit {pinned_commit[:10]} (sweep in progress)')
        subprocess.run(['git', 'checkout', pinned_commit], check=True)
    else:
        print(f'Already on pinned commit {pinned_commit[:10]}')
else:
    subprocess.run(['git', 'checkout', 'main'], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], check=True)
    new_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
    sweep_state['git_commit'] = new_commit
    _write_sweep_state(sweep_state)
    print(f'Pinned to commit {new_commit[:10]} for this sweep')

print('Commit:', subprocess.check_output(['git', 'log', '--oneline', '-1']).decode().strip())

In [ ]:
# Install only if not already set up (survives kernel restart within same runtime).
# Checks imports AND version compatibility, not just importability.
_REQUIRED_VERSIONS = {
    'hydra': ('hydra', '1.3'),
    'omegaconf': ('omegaconf', '2.3'),
    'h5py': ('h5py', '3.'),
    'torch': ('torch', '2.'),
}

def _deps_ok():
    """Return True only if all required packages import and meet version floors."""
    try:
        import hydra, h5py, networkx, statsmodels, tqdm, omegaconf, torch
        import superadditivity
        for mod_name, (pkg, prefix) in _REQUIRED_VERSIONS.items():
            ver = getattr(__import__(pkg), '__version__', '')
            if not ver.startswith(prefix):
                print(f'{pkg} version {ver} does not match required {prefix}*')
                return False
        return True
    except ImportError:
        return False

if _deps_ok():
    print('Packages already installed with compatible versions, skipping pip install.')
else:
    packages = ['hydra-core==1.3.2', 'omegaconf==2.3.0', 'h5py', 'networkx',
                'statsmodels', 'seaborn', 'tabulate', 'tqdm', 'scikit-learn', 'pandas', 'scipy', 'medmnist']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
subprocess.run([sys.executable, 'scripts/validate_setup.py'], check=True)

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import shutil

def link_drive_folder(local, drive):
    local, drive = Path(local), Path(drive)
    drive.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        if local.resolve() == drive.resolve():
            print(f'{local.name} -> {drive}')
            return
        local.unlink()
    elif local.exists():
        # Preserve local files and merge only missing files into Drive.
        for folder, _, files in os.walk(local):
            target = drive / Path(folder).relative_to(local)
            target.mkdir(parents=True, exist_ok=True)
            for filename in files:
                if not (target / filename).exists():
                    shutil.copy2(Path(folder) / filename, target / filename)
        backup = local.with_name(local.name + '.local-backup-' +
                                 datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        local.rename(backup)
        print(f'Local originals preserved at {backup}')
    local.symlink_to(drive, target_is_directory=True)
    print(f'{local.name} -> {drive}')

DRIVE_OUT = '/content/drive/MyDrive/superadditivity_outputs'
DRIVE_DATA = '/content/drive/MyDrive/superadditivity_data'
LOCAL_DATA = '/content/quantify_superadditivity/data'
link_drive_folder('/content/quantify_superadditivity/outputs', DRIVE_OUT)
link_drive_folder(LOCAL_DATA, DRIVE_DATA)

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime
import os
import shutil
import urllib.request
import zipfile
from tqdm.auto import tqdm

DATA_READY = False
DATA_VALIDATED = False
EXPECTED_COUNTS = {'train': 33525, 'test': 14604}
EXPECTED_CLASSES = 345
SPLIT_URL = 'https://csr.bu.edu/ftp/visda/2019/multi-source/domainnet/txt'
ARCHIVE_URL = 'https://csr.bu.edu/ftp/visda/2019/multi-source/groundtruth/clipart.zip'


def parse_split(text, split):
    entries, labels = {}, {}
    for number, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        fields = line.split()
        if len(fields) != 2:
            raise ValueError(f'{split}, line {number}: expected image path and label')
        path = PurePosixPath(fields[0])
        if (len(path.parts) != 3 or path.parts[0] != 'clipart'
                or any(p in ('.', '..') for p in path.parts)):
            raise ValueError(f'{split}, line {number}: incompatible path {path}')
        label = int(fields[1])
        cls = path.parts[1]
        if cls in labels and labels[cls] != label:
            raise ValueError(f'Conflicting labels for {cls}')
        labels[cls] = label
        relative = '/'.join(path.parts[1:])
        if relative in entries:
            raise ValueError(f'Duplicate entry: {relative}')
        entries[relative] = str(path)
    if len(entries) != EXPECTED_COUNTS[split] or len(labels) != EXPECTED_CLASSES:
        raise ValueError(f'{split}: {len(entries)} images / {len(labels)} classes; '
                         f'expected {EXPECTED_COUNTS[split]} / {EXPECTED_CLASSES}')
    if labels != {cls: i for i, cls in enumerate(sorted(labels))}:
        raise ValueError('Official labels do not match ImageFolder class ordering')
    return entries, labels


def fetch(url, destination):
    # A failed download never replaces an existing cache file.
    temporary = destination.with_name(destination.name + '.part')
    with urllib.request.urlopen(url, timeout=120) as response:
        total = int(response.headers.get('Content-Length', 0)) or None
        with temporary.open('wb') as out, tqdm(
            total=total, unit='B', unit_scale=True, desc=destination.name
        ) as progress:
            while chunk := response.read(1024 * 1024):
                out.write(chunk)
                progress.update(len(chunk))
        if total is not None and temporary.stat().st_size != total:
            raise IOError(f'Incomplete download: {destination.name}')
    temporary.replace(destination)


def prepare_domainnet(root):
    downloads = root / '_downloads'
    downloads.mkdir(parents=True, exist_ok=True)
    backup = root / '_repair_backups' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')

    def preserve(path):
        target = backup / path.relative_to(root)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(path), str(target))
        print(f'Preserved unexpected/invalid item: {target}')

    manifests, label_maps = {}, {}
    for split in EXPECTED_COUNTS:
        target = downloads / f'clipart_{split}.txt'
        try:
            manifests[split], label_maps[split] = parse_split(
                target.read_text(encoding='utf-8'), split)
        except (OSError, ValueError, UnicodeError) as error:
            print(f'Replacing {split} list: {error}')
            candidate = downloads / f'clipart_{split}.new.txt'
            fetch(f'{SPLIT_URL}/clipart_{split}.txt', candidate)
            manifests[split], label_maps[split] = parse_split(
                candidate.read_text(encoding='utf-8'), split)
            if target.exists():
                preserve(target)
            candidate.replace(target)
    if label_maps['train'] != label_maps['test']:
        raise RuntimeError('Train/test class labels differ')
    if set(manifests['train']) & set(manifests['test']):
        raise RuntimeError('Train/test lists overlap')

    missing = []
    for split, entries in manifests.items():
        destination = root / 'clipart' / split
        destination.mkdir(parents=True, exist_ok=True)
        classes = label_maps[split]
        actual_classes = {p.name for p in destination.iterdir() if p.is_dir()}
        print(f'{split}: {len(actual_classes)} class folders; expected {len(classes)}')
        print('  Missing classes:', sorted(set(classes) - actual_classes))
        print('  Unexpected classes:', sorted(actual_classes - set(classes)))
        # Move unexpected folders/files outside ImageFolder, never delete them.
        for child in list(destination.iterdir()):
            if child.name not in classes or not child.is_dir():
                preserve(child)
                continue
            for item in list(child.iterdir()):
                relative = f'{child.name}/{item.name}'
                if item.is_dir() or relative not in entries:
                    preserve(item)
        for relative, member in tqdm(entries.items(), desc=f'Checking {split}'):
            target = destination / relative
            if not target.is_file() or target.stat().st_size == 0:
                missing.append((target, member))
        print(f'  {sum(1 for p, _ in missing if destination in p.parents):,} images need repair')

    if not missing:
        print('Dataset complete: skipping all image downloads and copying.')
        return label_maps['train']

    unresolved = []
    raw = downloads / 'clipart_raw'
    for target, member in tqdm(missing, desc='Restoring from extracted cache'):
        source = raw / member
        if source.is_file() and source.stat().st_size > 0:
            target.parent.mkdir(parents=True, exist_ok=True)
            temp = target.with_name(target.name + '.part')
            shutil.copyfile(source, temp)
            temp.replace(target)
        else:
            unresolved.append((target, member))

    if unresolved:
        archive = downloads / 'clipart.zip'
        required = {member for _, member in unresolved}

        def archive_usable():
            try:
                with zipfile.ZipFile(archive) as z:
                    return required <= set(z.namelist())
            except (OSError, zipfile.BadZipFile):
                return False

        if not archive_usable():
            if archive.exists():
                preserve(archive)
            print('Downloading the official archive to restore missing source images.')
            fetch(ARCHIVE_URL, archive)
        if not archive_usable():
            raise RuntimeError('Archive lacks required images; preparation stopped.')
        # Read only missing images from ZIP; no full re-extraction is required.
        with zipfile.ZipFile(archive) as z:
            for target, member in tqdm(unresolved, desc='Restoring from ZIP'):
                target.parent.mkdir(parents=True, exist_ok=True)
                temp = target.with_name(target.name + '.part')
                with z.open(member) as source, temp.open('wb') as out:
                    shutil.copyfileobj(source, out)
                temp.replace(target)

    for split, entries in manifests.items():
        destination = root / 'clipart' / split
        failures = [r for r in entries if not (destination / r).is_file()
                    or (destination / r).stat().st_size == 0]
        if failures:
            raise RuntimeError(f'{split}: {len(failures)} images still missing/empty: {failures[:10]}')
        print(f'{split}: {len(entries):,} images, {len(label_maps[split])} classes verified')
    return label_maps['train']


root = Path(LOCAL_DATA) / 'domainnet'
expected_class_to_idx = prepare_domainnet(root)
DATA_READY = True

In [ ]:
DATA_VALIDATED = False
if not globals().get('DATA_READY', False):
    raise RuntimeError('Run the dataset preparation cell successfully first.')

from torchvision.datasets import ImageFolder
from superadditivity.datasets.dataset_loader import DatasetLoader

for split, expected_count in EXPECTED_COUNTS.items():
    dataset = ImageFolder(str(root / 'clipart' / split))
    if dataset.class_to_idx != expected_class_to_idx:
        raise RuntimeError(f'{split}: class mapping differs from official labels; '
                           f'found {len(dataset.classes)} classes, expected {EXPECTED_CLASSES}')
    if len(dataset) != expected_count:
        raise RuntimeError(f'{split}: found {len(dataset)} images; expected {expected_count}')
    print(f'{split}: {len(dataset):,} images, {len(dataset.classes)} classes')

loader = DatasetLoader(dataset_name='domainnet', data_dir=str(root), download=False)
train_ds, test_ds = loader.load()
validation_output = os.path.join(DRIVE_OUT, 'domainnet_validation')
os.makedirs(validation_output, exist_ok=True)
loader.validate_and_visualize(num_samples=5, output_dir=validation_output)
DATA_VALIDATED = True
print(f'Validation passed. Sample images: {validation_output}')

In [ ]:
SWEEP_COMPLETE = False
if not globals().get('DATA_READY', False) or not globals().get('DATA_VALIDATED', False):
    raise RuntimeError('Complete dataset preparation and validation before the sweep.')
print('Starting sweep with --resume: completed runs will be skipped, interrupted runs resumed from checkpoint.')
print('If any experiment fails, the sweep will STOP immediately.')
result = subprocess.run([sys.executable, 'scripts/run_sweep.py', '--experiment',
                         'pout_sweep_domainnet', '--resume'])
if result.returncode != 0:
    raise RuntimeError(
        f'Sweep failed with exit code {result.returncode}. '
        f'Check the error.json in the failed run directory and the console output above. '
        f'Fix the issue, then click "Run all" to resume from where it stopped.'
    )
SWEEP_COMPLETE = True
print('Sweep finished. All 16 configurations complete.')

In [ ]:
ANALYSIS_COMPLETE = False
if not globals().get('SWEEP_COMPLETE', False):
    raise RuntimeError(
        'Sweep did not complete successfully. Fix the sweep errors before running analysis. '
        'Check error.json in the failed run directory for details.'
    )

# Analysis and visualization skip automatically when inputs are unchanged.
# Add --force to either command to regenerate.
result = subprocess.run([sys.executable, 'scripts/run_analysis.py', '--results_dir',
                         'outputs/pout_sweep_domainnet', '--analysis', 'surface'])
if result.returncode != 0:
    raise RuntimeError(
        f'Analysis failed with exit code {result.returncode}. '
        f'Check the console output above for details.'
    )

result = subprocess.run([sys.executable, 'scripts/run_visualization.py', '--results_dir',
                         'outputs/pout_sweep_domainnet'])
if result.returncode != 0:
    raise RuntimeError(
        f'Visualization failed with exit code {result.returncode}. '
        f'Check the console output above for details.'
    )

ANALYSIS_COMPLETE = True
print('Analysis and visualization complete.')